In [ ]:
import rasterio
import numpy as np
from rasterio.merge import merge
import os

In [ ]:
# Read all files in predicted folder

file_path = "../output/predicted/"
file_list = [f for f in os.listdir(nonan_folder)]

In [ ]:
datasets = [rasterio.open(file) for file in file_list]
data_arrays = [ds.read(1) for ds in datasets]  
# Đọc band đầu tiên của mỗi file
datasets

In [ ]:
for i, ds in enumerate(datasets):
    print(f"File {file_list[i]}: {ds.count} bands")

In [ ]:
# Ghép dữ liệu: với mỗi band, lấy giá trị max từ 6 file
combined_bands = []
num_bands = 13  # Số band mỗi file
for band_idx in range(1, num_bands + 1):
    # Đọc band hiện tại từ tất cả 6 file
    band_data = [ds.read(band_idx) for ds in datasets]
    # Chuyển thành mảng 3D (6 file, height, width)
    band_stack = np.stack(band_data, axis=0)
    # Tính giá trị max theo trục file (trục 0)
    band_max = np.max(band_stack, axis=0)
    combined_bands.append(band_max)

In [ ]:
# Chuyển danh sách thành mảng 3D (13 band, height, width)
combined_bands = np.stack(combined_bands, axis=0)

In [ ]:
meta = datasets[0].meta.copy()
meta.update({
    "count": num_bands,  # File đầu ra có 13 band
    "height": combined_bands.shape[1],
    "width": combined_bands.shape[2]
})

In [ ]:
with rasterio.open("./output/cloud_concat/combined_13bands.tif", "w", **meta) as dest:
    for band_idx in range(num_bands):
        dest.write(combined_bands[band_idx], band_idx + 1)

# Đóng file
for ds in datasets:
    ds.close()